# Exp-3 — Qwen3-32B HyDE + Enumeration

**Why this exp.** Exp-2 with Qwen2.5-7B reached `hyde+enum stat_recall@500 = 29.5%` (gate: 40%). Diagnostic showed 7B hallucinates Swiss article numbers — only 7/251 gold appear verbatim. Tests here whether a bigger, newer model (Qwen3-32B) fixes the knowledge problem with everything else held constant.

**Identical to Exp-2** apart from:
- Model: `Qwen/Qwen3-32B` in bf16 (full precision, ~64 GB weights + kv-cache).
- Requires a GPU with ≥ 80 GB VRAM (user's 95 GB setup fits comfortably).
- Qwen3's built-in *thinking mode* is **disabled** (`enable_thinking=False`) — we want strict HyDE paragraphs and citation lines, not reasoning traces.
- Same prompts, same 8 fusion variants reported side-by-side.

**Gate**: `stat_recall@500 ≥ 0.40`. Also inspect whether the hallucination rate drops (verbatim-gold-in-enum should rise noticeably from 7/251 if Qwen3-32B actually knows Swiss law).

Based on the troubleshooting steps we went through, here is the correct and most stable sequence for installing the libraries in your Colab environment:

Install vllm first: Since vLLM has strict dependencies (especially tied to PyTorch and CUDA versions), it's best to install it before other libraries so its dependency resolver can set up the core packages.

!pip install vllm
(Note: If you encounter ABI errors, you use the --extra-index-url pointing to your specific PyTorch/CUDA version as seen in the troubleshooting cells).

Upgrade transformers: Installing vLLM can sometimes downgrade or mess up the transformers installation, leading to the ModuleNotFoundError: No module named 'transformers.image_processing_backends' error we saw. Upgrading it immediately after fixes this:

!pip install --upgrade transformers -q
Install FlagEmbedding and data tools: Install your embedding and data manipulation libraries next.

!pip install -qU FlagEmbedding pandas numpy
Apply runtime patches (Python-side): Because we upgraded transformers (which removed is_torch_fx_available and changed how jina_embeddings_v3 is loaded), FlagEmbedding will fail to import natively. You must include the dynamic patches we added in Cell 8 before importing FlagEmbedding:

import sys
import types
import transformers.utils.import_utils

# Patch missing functions/modules for FlagEmbedding
if not hasattr(transformers.utils.import_utils, 'is_torch_fx_available'):
    transformers.utils.import_utils.is_torch_fx_available = lambda: False
if 'transformers.models.jina_embeddings_v3' not in sys.modules:
    sys.modules['transformers.models.jina_embeddings_v3'] = types.ModuleType('transformers.models.jina_embeddings_v3')
By following this exact order, you satisfy vLLM's strict requirements, fix the broken transformers installation, and successfully bypass FlagEmbedding's import errors!


In [4]:
# --- Cell 1. Install deps ---
# vLLM >= 0.8.5 ships Qwen3 support. No AWQ needed — running bf16 on 95 GB GPU.
!pip uninstall -y FlagEmbedding
!pip install -qU FlagEmbedding pandas numpy

Found existing installation: FlagEmbedding 1.2.11
Uninstalling FlagEmbedding-1.2.11:
  Successfully uninstalled FlagEmbedding-1.2.11
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 162.5 MB/s eta 0:00:00


In [ ]:
# Install vLLM separately to analyze potential errors more clearly
!pip install vllm

In [9]:
# --- Cell 2. Mount Drive & paths (reuses Exp-1/2 artifacts) ---
from google.colab import drive
drive.mount('/content/drive')

import json, pickle, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/swiss_law/data')
ART  = ROOT / 'artifacts'
assert (ART / 'laws_bgem3.npy').exists(), 'run Exp-1 first (needs laws_bgem3.npy)'

# Verify GPU fits Qwen3-32B in bf16 (~64 GB weights + kv-cache).
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {name} ({vram:.0f} GB)')
assert vram >= 78, 'need >= 80 GB VRAM to run Qwen3-32B bf16 (user reports 95 GB)'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (95 GB)


In [10]:
# --- Cell 3. Load val + corpus metadata ---
val  = pd.read_csv(ROOT / 'val.csv')
laws = pd.read_csv(ROOT / 'laws_de.csv')
with open(ROOT / 'val_translated_de.pkl', 'rb') as f:
    val_de = pickle.load(f)
cits = laws['citation'].tolist()
doc_emb = np.load(ART / 'laws_bgem3.npy').astype(np.float32)
print('corpus:', doc_emb.shape, '| val:', len(val))

corpus: (175933, 1024) | val: 10


In [6]:
import os
os.environ['VLLM_NO_USAGE_STATS'] = '1'
print('VLLM_NO_USAGE_STATS set to 1')

VLLM_NO_USAGE_STATS set to 1


In [15]:
# --- Cell 4. Load Qwen3-32B via vLLM (bf16, thinking mode off) ---
!pip install --upgrade transformers -q

import os
import sys

os.environ["VLLM_NO_USAGE_STATS"] = "1"

# Workaround for io.UnsupportedOperation: fileno in Colab
# Override explicitly because Colab's stream has the method but it raises an error
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2
from vllm import LLM, SamplingParams

llm = LLM(model='Qwen/Qwen3-32B',
          dtype='bfloat16',
          gpu_memory_utilization=0.90,
          max_model_len=4096,
          enforce_eager=False)

# Qwen3 enables "thinking" by default (emits <think>...</think> before the answer).
# For strict HyDE/Enum output we disable it via chat_template_kwargs.
CHAT_KW = {'enable_thinking': False}

def chat(messages, **kw):
    params = SamplingParams(temperature=kw.get('temperature', 0.2),
                            top_p=0.9,
                            max_tokens=kw.get('max_tokens', 800))
    out = llm.chat(messages,
                   sampling_params=params,
                   chat_template_kwargs=CHAT_KW,
                   use_tqdm=False)
    return out[0].outputs[0].text.strip()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 3.1.1 requires transformers<5.0.0,>=4.38.0, but you have transformers 5.5.4 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
INFO 04-21 15:04:40 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'max_model_len': 4096, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-32B'}
INFO 04-21 15:04:41 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 04-21 15:04:41 [model.py:1678] Using max model len 4096
INFO 04-21 15:04:41 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 04-21 15:04:41 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-21 15:04:43 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See 

In [5]:
import torch

print(f"Installed PyTorch version: {torch.__version__}")
print(f"Installed CUDA version: {torch.version.cuda}")

Installed PyTorch version: 2.10.0+cu128
Installed CUDA version: 12.8


The error you encountered (RuntimeError related to `_core_C.ScalarType`) typically arises from an ABI incompatibility between `vLLM` and your `PyTorch` installation. `vLLM` often requires specific `PyTorch` versions that match the CUDA version it was compiled against.

To find a compatible `vLLM` build, you should:

1.  **Note your installed PyTorch and CUDA versions** from the output of the cell above.
2.  **Visit the official `vLLM` documentation or their GitHub releases page** (e.g., their PyPI project page).
3.  **Look for pre-built `vLLM` wheels (packages)** that explicitly state compatibility with your specific PyTorch and CUDA versions (e.g., a package named `vllm-x.y.z+cu121-torch2.1`).

Once you've identified a compatible wheel, you will likely need to uninstall your current `vLLM` and then reinstall the correct version. A common installation pattern looks like this:

```python
!pip uninstall -y vllm
!pip install vllm==[COMPATIBLE_VLLM_VERSION] --extra-index-url https://download.vllm.ai/whl/[TORCH_VERSION]+[CUDA_VERSION]
```

For example, if you have `PyTorch 2.1.0` and `CUDA 12.1`, it might be:

```python
!pip uninstall -y vllm
!pip install vllm==0.4.0 --extra-index-url https://download.vllm.ai/whl/torch2.1.0+cu121
```

(Remember to replace `[COMPATIBLE_VLLM_VERSION]`, `[TORCH_VERSION]`, and `[CUDA_VERSION]` with the versions you find to be compatible for your setup.)

In [ ]:
# Uninstall the current vLLM installation
!pip uninstall -y vllm

# Attempt to install vLLM compatible with PyTorch 2.10.0 and CUDA 12.8
# We let pip find the latest vLLM version available for this specific PyTorch/CUDA wheel index.
# Note: vLLM typically builds against standard PyTorch releases (e.g., 2.1, 2.2, 2.3, 2.4).
# 'torch2.10.0' might be a Colab-specific or unusual version, so a direct match might not exist.
!pip install vllm --extra-index-url https://download.vllm.ai/whl/torch2.10.0+cu128

In [ ]:
# Uninstall the current vLLM installation
!pip uninstall -y vllm

# Attempt to install vLLM compatible with PyTorch 2.4.0 and CUDA 12.8
# This matches a version mentioned in earlier pip dependency warnings and is a common vLLM build target.
!pip install vllm --extra-index-url https://download.vllm.ai/whl/torch2.4.0+cu128

After running the above cell, please try running Cell 4 (`MLGPYVHpUvz6`) again to see if the `vLLM` initialization error is resolved. If this installation command also fails to find a suitable wheel or if the `vLLM` error persists, you will need to consult the official `vLLM` documentation or their GitHub releases page for the exact compatible `PyTorch` and `CUDA` combinations for the latest `vLLM` versions.

After running the above cell, please try running Cell 4 (`MLGPYVHpUvz6`) again to see if the `vLLM` initialization error is resolved. If the `vLLM` installation command fails due to not finding a wheel, it means a pre-built package for `torch2.10.0+cu128` isn't available. In that case, you might consider trying `torch2.4.0+cu128` in the `--extra-index-url` (as that version was mentioned in `pip`'s dependency warnings), or check the official `vLLM` documentation for compatible versions.

In [12]:
# --- Cell 5. Prompts (identical to Exp-2, for a clean A/B on model size) ---
HYDE_SYS = (
    'Du bist Schweizer Jurist. Schreibe deutsche Texte im Stil Schweizer Bundesgesetze.'
)
HYDE_USER = (
    'Gegeben ist folgender Sachverhalt (auf Englisch):\n\n{q}\n\n'
    'Schreibe DREI hypothetische deutsche Paragraphen im Stil Schweizer Bundesgesetze '
    '(OR, ZGB, StGB, StPO, ZPO, BGG, etc.), die diesen Sachverhalt rechtlich regeln. '
    'Jeder Paragraph 2-4 Sätze. Formal-juristischer Ton. '
    'Trenne die drei Paragraphen mit \"---\". Keine Überschriften, kein Vorspann, '
    'nur die drei Paragraphen.'
)

ENUM_SYS = (
    'You are a Swiss legal expert. You know the Swiss Federal Codes (OR, ZGB, StGB, StPO, '
    'ZPO, BGG, BV, IPRG, SchKG, DBG, StHG, etc.) and typical articles cited for common '
    'legal issues.'
)
ENUM_USER = (
    'Scenario:\n\n{q}\n\n'
    'List 15 Swiss federal law citations a Swiss lawyer would most likely consult to analyse '
    'this case. Use exact format \"Art. X Abs. Y ABBR\" or \"Art. X ABBR\" (e.g. \"Art. 397 '
    'Abs. 1 OR\", \"Art. 2 ZGB\"). One citation per line, followed by a 4-8 word reason '
    'separated by \" — \". No numbering, no headers, no extra commentary.'
)

def build_prompts(q_en):
    return (
        [{'role': 'system', 'content': HYDE_SYS},
         {'role': 'user',   'content': HYDE_USER.format(q=q_en)}],
        [{'role': 'system', 'content': ENUM_SYS},
         {'role': 'user',   'content': ENUM_USER.format(q=q_en)}],
    )

In [16]:
# --- Cell 6. Generate expansions ---
expansions = {}
t0 = time.time()
for row in val.itertuples():
    q = row.query
    hyde_msgs, enum_msgs = build_prompts(q)
    hyde_txt = chat(hyde_msgs, temperature=0.3, max_tokens=700)
    enum_txt = chat(enum_msgs, temperature=0.1, max_tokens=600)
    expansions[row.query_id] = {
        'query_en': q,
        'query_de': val_de[row.query_id],
        'hyde': hyde_txt,
        'enum': enum_txt,
    }
    print(f'{row.query_id}: HyDE {len(hyde_txt)}c | Enum {len(enum_txt)}c')
print(f'total gen time: {time.time() - t0:.1f}s')

with open(ART / 'exp_A3_expansions.json', 'w') as f:
    json.dump(expansions, f, ensure_ascii=False, indent=2)

# quick hallucination check vs gold
def parse(s): return [c.strip() for c in str(s).split(';') if c.strip()]
def is_stat(c):
    return not (c.startswith('BGE ') or re.match(r'\d[A-Z]_', c) or re.match(r'[A-Z]\d[A-Z]_', c))

hit_verbatim = hit_total = 0
for row in val.itertuples():
    golds = [c for c in parse(row.gold_citations) if is_stat(c)]
    enum = expansions[row.query_id]['enum']
    hit_verbatim += sum(1 for g in golds if g in enum)
    hit_total    += len(golds)
print(f'\nVerbatim gold-in-enum: {hit_verbatim}/{hit_total} = {100*hit_verbatim/hit_total:.1f}% '
      f'(Exp-2 / 7B was 7/251 = 2.8%)')

INFO 04-21 15:05:36 [hf.py:314] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
val_001: HyDE 1421c | Enum 760c
val_002: HyDE 1790c | Enum 694c
val_003: HyDE 1320c | Enum 1045c
val_004: HyDE 1539c | Enum 620c
val_005: HyDE 1510c | Enum 757c
val_006: HyDE 1538c | Enum 973c
val_007: HyDE 1378c | Enum 751c
val_008: HyDE 1619c | Enum 684c
val_009: HyDE 1597c | Enum 744c
val_010: HyDE 1586c | Enum 809c
total gen time: 319.5s

Verbatim gold-in-enum: 6/149 = 4.0% (Exp-2 / 7B was 7/251 = 2.8%)


In [17]:
# --- Cell 7. Free the Qwen GPU memory ---
import gc
del llm
gc.collect(); torch.cuda.empty_cache()
print(f'free vram: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB')

free vram: 7.8 GB


In [18]:
# --- Cell 8. Encode queries + expansions with BGE-M3 ---
import sys
import types
import transformers.utils.import_utils

# Patch 1: Mock is_torch_fx_available
if not hasattr(transformers.utils.import_utils, 'is_torch_fx_available'):
    transformers.utils.import_utils.is_torch_fx_available = lambda: False

# Patch 2: Mock missing jina_embeddings_v3 module
if 'transformers.models.jina_embeddings_v3' not in sys.modules:
    sys.modules['transformers.models.jina_embeddings_v3'] = types.ModuleType('transformers.models.jina_embeddings_v3')

from typing import Optional
from FlagEmbedding import BGEM3FlagModel
enc = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

def embed(texts, max_len=2048):
    return enc.encode(texts, batch_size=4, max_length=max_len,
                      return_dense=True, return_sparse=False,
                      return_colbert_vecs=False)['dense_vecs']

en_qs   = [expansions[r.query_id]['query_en'] for r in val.itertuples()]
de_qs   = [expansions[r.query_id]['query_de'] for r in val.itertuples()]
hyde_qs = [expansions[r.query_id]['hyde']     for r in val.itertuples()]
enum_qs = [expansions[r.query_id]['enum']     for r in val.itertuples()]

q_en   = embed(en_qs)
q_de   = embed(de_qs)
q_hyde = embed(hyde_qs)
q_enum = embed(enum_qs, max_len=1024)

np.savez(ART / 'query_vecs_A3.npz',
         q_en=q_en, q_de=q_de, q_hyde=q_hyde, q_enum=q_enum,
         query_ids=np.array([r.query_id for r in val.itertuples()]))


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 3/3 [00:00<00:00, 178.08it/s]


In [19]:
# --- Cell 9. Retrieval + RRF fusion (identical to Exp-2) ---
def rank_all(q):
    sims = q @ doc_emb.T
    return np.argsort(-sims, axis=1)

ranks = {
    'en':   rank_all(q_en),
    'de':   rank_all(q_de),
    'hyde': rank_all(q_hyde),
    'enum': rank_all(q_enum),
}

def rrf(rank_lists, k_rrf=60, topk=2000):
    N_q, N_doc = rank_lists[0].shape
    scores = np.zeros((N_q, N_doc), dtype=np.float32)
    for rl in rank_lists:
        pos = np.empty_like(rl)
        rows = np.arange(N_q)[:, None]
        pos[rows, rl] = np.arange(N_doc)[None, :]
        scores += 1.0 / (k_rrf + pos.astype(np.float32))
    idx = np.argpartition(-scores, topk - 1, axis=1)[:, :topk]
    rows = np.arange(N_q)[:, None]
    order = np.argsort(-scores[rows, idx], axis=1)
    return idx[rows, order]

def eval_ranking(top_idx, label, ks=(50, 200, 500, 1000)):
    per_q = []
    for i, row in enumerate(val.itertuples()):
        gold_stat = {c for c in parse(row.gold_citations) if is_stat(c)}
        retrieved = [cits[j] for j in top_idx[i]]
        entry = {'query_id': row.query_id, 'n_gold_stat': len(gold_stat)}
        for k in ks:
            entry[f'stat_hit@{k}'] = len(gold_stat & set(retrieved[:k]))
        per_q.append(entry)
    agg = {f'stat_recall@{k}': sum(p[f'stat_hit@{k}'] for p in per_q)
                              / max(1, sum(p['n_gold_stat'] for p in per_q)) for k in ks}
    print(f'=== {label} ===')
    for k, v in agg.items():
        print(f'  {k} = {v:.3f}')
    return {'agg': agg, 'per_query': per_q}

report = {}
report['en_only']      = eval_ranking(ranks['en'][:, :1000], 'EN only')
report['hyde_only']    = eval_ranking(ranks['hyde'][:, :1000], 'HyDE only')
report['enum_only']    = eval_ranking(ranks['enum'][:, :1000], 'Enum only')
report['en+hyde']      = eval_ranking(rrf([ranks['en'], ranks['hyde']]), 'RRF(en, hyde)')
report['en+enum']      = eval_ranking(rrf([ranks['en'], ranks['enum']]), 'RRF(en, enum)')
report['hyde+enum']    = eval_ranking(rrf([ranks['hyde'], ranks['enum']]), 'RRF(hyde, enum)')
report['en+hyde+enum'] = eval_ranking(rrf([ranks['en'], ranks['hyde'], ranks['enum']]), 'RRF(en, hyde, enum)')
report['all4']         = eval_ranking(rrf([ranks['en'], ranks['de'], ranks['hyde'], ranks['enum']]), 'RRF(en, de, hyde, enum)')

=== EN only ===
  stat_recall@50 = 0.074
  stat_recall@200 = 0.121
  stat_recall@500 = 0.215
  stat_recall@1000 = 0.302
=== HyDE only ===
  stat_recall@50 = 0.148
  stat_recall@200 = 0.262
  stat_recall@500 = 0.322
  stat_recall@1000 = 0.369
=== Enum only ===
  stat_recall@50 = 0.107
  stat_recall@200 = 0.242
  stat_recall@500 = 0.315
  stat_recall@1000 = 0.403
=== RRF(en, hyde) ===
  stat_recall@50 = 0.134
  stat_recall@200 = 0.262
  stat_recall@500 = 0.302
  stat_recall@1000 = 0.383
=== RRF(en, enum) ===
  stat_recall@50 = 0.094
  stat_recall@200 = 0.208
  stat_recall@500 = 0.315
  stat_recall@1000 = 0.416
=== RRF(hyde, enum) ===
  stat_recall@50 = 0.161
  stat_recall@200 = 0.315
  stat_recall@500 = 0.376
  stat_recall@1000 = 0.456
=== RRF(en, hyde, enum) ===
  stat_recall@50 = 0.174
  stat_recall@200 = 0.315
  stat_recall@500 = 0.369
  stat_recall@1000 = 0.436
=== RRF(en, de, hyde, enum) ===
  stat_recall@50 = 0.128
  stat_recall@200 = 0.275
  stat_recall@500 = 0.356
  stat_recall@1

In [20]:
# --- Cell 10. Save report + A/B vs Exp-2 ---
report['meta'] = {'model_retriever': 'BAAI/bge-m3',
                  'model_llm': 'Qwen/Qwen3-32B',
                  'llm_thinking': False,
                  'n_queries': len(val),
                  'exp2_7b_best_recall@500': 0.295,
                  'gate': 'stat_recall@500 >= 0.40'}
with open(ART / 'exp_A3_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)

print('=' * 60)
print(f"{'variant':<20} {'rec@50':>7} {'@200':>7} {'@500':>7} {'@1000':>7}")
print('-' * 60)
for k in ['en_only','hyde_only','enum_only','en+hyde','en+enum','hyde+enum','en+hyde+enum','all4']:
    a = report[k]['agg']
    print(f"{k:<20} {a['stat_recall@50']:>7.3f} {a['stat_recall@200']:>7.3f} "
          f"{a['stat_recall@500']:>7.3f} {a['stat_recall@1000']:>7.3f}")

best_key = max((k for k in report if k != 'meta'),
               key=lambda k: report[k]['agg']['stat_recall@500'])
best = report[best_key]['agg']['stat_recall@500']
print(f"\nbest: {best_key} -> stat_recall@500 = {best:.3f}")
print(f"vs Exp-2 (Qwen2.5-7B) best = 0.295; gate 0.40 -> {'PASS' if best >= 0.40 else 'STILL BELOW'}")
print(f"vs Exp-1 (no LLM)          = 0.215")

variant               rec@50    @200    @500   @1000
------------------------------------------------------------
en_only                0.074   0.121   0.215   0.302
hyde_only              0.148   0.262   0.322   0.369
enum_only              0.107   0.242   0.315   0.403
en+hyde                0.134   0.262   0.302   0.383
en+enum                0.094   0.208   0.315   0.416
hyde+enum              0.161   0.315   0.376   0.456
en+hyde+enum           0.174   0.315   0.369   0.436
all4                   0.128   0.275   0.356   0.409

best: hyde+enum -> stat_recall@500 = 0.376
vs Exp-2 (Qwen2.5-7B) best = 0.295; gate 0.40 -> STILL BELOW
vs Exp-1 (no LLM)          = 0.215
